# BERT: Pre-training of Deep Bidirectional Transformers - 실습 코드 1: BERT MLM 파인튜닝 (HuggingFace)

- Tutorial ID: `expand-bert-paper`
- Tutorial: BERT: Pre-training of Deep Bidirectional Transformers
- Section ID: `expand-bert-paper-code-1`
- Section: 실습 코드 1: BERT MLM 파인튜닝 (HuggingFace)


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 1: BERT MLM 파인튜닝 (HuggingFace)
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) logit이 확률분포로 바뀌는 과정과 temperature/top-k/top-p의 효과 관찰
#   2) 미래 토큰을 -inf로 막은 뒤 softmax 확률이 0이 되는지 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
from transformers import BertTokenizer, BertForMaskedLM
import torch

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForMaskedLM.from_pretrained('bert-base-uncased')

# 마스크된 문장 입력
text = "The transformer architecture replaced [MASK] in NLP."
inputs = tokenizer(text, return_tensors='pt')

# [MASK] 위치 찾기
mask_token_id = tokenizer.mask_token_id
mask_idx = (inputs.input_ids == mask_token_id).nonzero()[0, 1]

# 예측
with torch.no_grad():
    outputs = model(**inputs)
        logits = outputs.logits

# Top-5 예측
probs = torch.softmax(logits[0, mask_idx], dim=-1)
top5 = torch.topk(probs, 5)
for i, (score, token_id) in enumerate(zip(top5.values, top5.indices)):
    token = tokenizer.decode([token_id])
    print(f"{i+1}. {token.strip()} ({score:.4f})")